In [8]:
#imports
import os
import io
from pandas import DataFrame
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

In [9]:
# Função responsável por percorrer e ler os arquivos de e-mail
def readFiles(path):
    # Percorre todos os diretórios e arquivos do caminho informado
    for root, dirnames, filenames in os.walk(path):
        for filename in filenames:
            # Monta o caminho completo do arquivo
            filePath = os.path.join(root, filename)

            inBody = False  # Indica se a leitura já chegou ao corpo do e-mail
            lines = []      # Lista que armazenará as linhas do corpo do e-mail

            # Abre o arquivo para leitura
            f = io.open(filePath, 'r', encoding='latin1')

            # Percorre cada linha do arquivo
            for line in f:
                # Se já estiver no corpo do e-mail, armazena a linha
                if inBody:
                    lines.append(line)

                # Uma linha vazia indica o fim do cabeçalho e início do corpo
                elif line == '\n':
                    inBody = True

            # Fecha o arquivo após a leitura
            f.close()

            # Junta todas as linhas do corpo em uma única string
            message = '\n'.join(lines)

            # Retorna o caminho do arquivo e sua mensagem
            yield filePath, message


# Cria um DataFrame a partir dos arquivos de um diretório
def dataFrameFromDirectory(path, classification):
    rows = []   # Lista que armazenará os dados dos e-mails
    index = []  # Lista que armazenará os nomes/caminhos dos arquivos

    # Lê todos os arquivos do diretório
    for filename, message in readFiles(path):
        # Adiciona a mensagem e sua classificação (spam ou ham)
        rows.append({
            'message': message,
            'class': classification
        })

        # Salva o nome/caminho do arquivo como índice
        index.append(filename)

    # Retorna um DataFrame contendo as mensagens e suas classes
    return DataFrame(rows, index=index)


# Cria um DataFrame vazio
data = DataFrame({'message': [], 'class': []})

# Junta os e-mails classificados como spam e ham em um único DataFrame
data = pd.concat([
    dataFrameFromDirectory(
        '/content/drive/MyDrive/Colab Notebooks/Card 13/emails/emails/spam',
        'spam'
    ),
    dataFrameFromDirectory(
        '/content/drive/MyDrive/Colab Notebooks/Card 13/emails/emails/ham',
        'ham'
    )
], ignore_index=True)

# Exibe as primeiras linhas do DataFrame
print(data.head())

                                             message class
0  Once upon a time, QuaffA wrote :\n\n\n\n> I've...   ham
1  On Mon, 2002-10-07 at 09:56, Matthias Saou wro...   ham
2  I can't seem to build this package. It errors ...   ham
3  Matthias Saou (matthias@rpmforge.net) wrote*:\...   ham
4  checking build system type... i686-pc-linux-gn...   ham


In [10]:
data.head() #pega o começo do data

,message,class
0,"Once upon a time, QuaffA wrote :\n\n\n\n> I've...",ham
1,"On Mon, 2002-10-07 at 09:56, Matthias Saou wro...",ham
2,I can't seem to build this package. It errors ...,ham
3,Matthias Saou (matthias@rpmforge.net) wrote*:\...,ham
4,checking build system type... i686-pc-linux-gn...,ham


In [12]:
vectorizer = CountVectorizer() #faz a contagem das palavras
counts = vectorizer.fit_transform(data['message'].values) #pega os emails e manda para um vetor

classifier = MultinomialNB() #modelo de classificação
targets = data['class'].values #pega a classe, ham ou spam e joga pra um array
classifier.fit(counts, targets) #treina

MultinomialNB()

In [14]:
examples = ['Free Viagra now!!!', 'Hi Bob, how about a game of golf tomorrow?'] #exemplo de texto de emal
example_counts = vectorizer.transform(examples)
predictions = classifier.predict(example_counts) # aqui é feito o teste  do modelo
predictions #Mostra como classificou cada um

array(['ham', 'ham'], dtype='<U3')